In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd


# Load your dataset
df = pd.read_csv('type_classification-train.csv')

# Assuming your dataset has columns: 'Sentence', 'Label 1', 'Label 2', 'Label 3'
sentences = df['sentence'].values
labels = df[['structure_focus','process_focus', 'usecase_focus']].values

# Split the dataset into training and testing sets
train_sentences, test_sentences, train_labels, test_labels = train_test_split(sentences, labels, test_size=0.2, random_state=42)

# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the sentences
train_encodings = tokenizer(train_sentences.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
test_encodings = tokenizer(test_sentences.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')

# Convert labels to tensors
train_labels = torch.tensor(train_labels, dtype=torch.float)
test_labels = torch.tensor(test_labels, dtype=torch.float)


class BERTLAM(nn.Module):
    def __init__(self, num_labels):
        super(BERTLAM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.label_attention = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # Label Attention Mechanism
        label_scores = self.label_attention(sequence_output)
        attention_weights = torch.softmax(label_scores, dim=1)
        weighted_sequence_output = torch.sum(sequence_output * attention_weights, dim=1)

        # Classifier
        logits = self.classifier(weighted_sequence_output)
        probs = self.sigmoid(logits)

        return probs
    
# Initialize the model
model = BERTLAM(num_labels=3)
optimizer = optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.BCELoss()

# Create DataLoader
train_dataset = torch.utils.data.TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)

# Training loop
model.train()
for epoch in range(3):  # Number of epochs
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch + 1}, Loss: {loss.item()}')
    


C:\Users\RAYMOND\AppData\Local\pypoetry\Cache\virtualenvs\text2uml-yWP-Bm0a-py3.9\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the chec

RuntimeError: The size of tensor a (768) must match the size of tensor b (3) at non-singleton dimension 2

In [ ]:
# Create DataLoader for test data
test_dataset = torch.utils.data.TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], test_labels)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

# Evaluation loop
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        outputs = model(input_ids, attention_mask)
        preds = (outputs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate F1 score
f1 = f1_score(all_labels, all_preds, average='micro')
print(f'Test F1 Score: {f1}')

In [ ]:
torch.save(model.state_dict(), 'bert_lam_model.pth')

In [ ]:
model = BERTLAM(num_labels=3)
model.load_state_dict(torch.load('bert_lam_model.pth'))
model.eval()